In [ ]:
!pip install torch xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2

In [ ]:
from unsloth import FastLanguageModel
import torch

from transformers import Trainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 蒸馏训练
## 1. 蒸馏模型 Trainer

In [ ]:
# 计算反向kl散度
def compute_rkl(
        logits, 
        teacher_logits, 
        target, 
        padding_id,
        reduction="sum", 
        temp = 1.0
    ):
        logits = logits / temp
        teacher_logits = teacher_logits / temp

        probs = torch.softmax(logits, -1, dtype=torch.float32)
        log_probs = torch.log_softmax(logits, -1, dtype=torch.float32)
        teacher_log_probs = torch.log_softmax(teacher_logits, -1, dtype=torch.float32)
        kl = (probs * (log_probs - teacher_log_probs))
        kl = kl.sum(-1)
        
        pad_mask = target.eq(padding_id)
        kl = kl.masked_fill(pad_mask, 0.0)
        if reduction == "sum":
            kl = kl.sum(dim=1)
        elif reduction == "mean":
            kl = kl.sum(dim=1) / (~pad_mask).sum(dim=1)
        
        return kl

In [ ]:
class OPDTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        
    @torch.no_grad()
    def generate_sequences(self, input_ids, attention_mask):
        
        self.model.eval()
        sequences = self.model.generate(input_ids=input_ids, 
                                      attention_mask=attention_mask,
                                      max_new_tokens=50,
                                      do_sample=True,
                                      temperature=1.0,
                                      pad_token_id=self.processing_class.pad_token_id,
                                      eos_token_id=self.processing_class.eos_token_id
                                      )
        
        torch.cuda.empty_cache()
        self.model.train()
        return sequences
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        prompt_ids = inputs["input_ids"].to(model.device)
        prompt_mask = inputs["attention_mask"].to(self.model.device)
        sequences = self.generate_sequences(prompt_ids, prompt_mask)
        attention_mask = (sequences != self.processing_class.pad_token_id).long()
        
        logits = model(sequences, attention_mask=attention_mask).logits[:, prompt_ids.shape[-1]:]
        
        with torch.no_grad():
            torch.cuda.empty_cache()
            teacher_outputs = self.teacher_model(sequences, attention_mask=attention_mask)
            teacher_logits = teacher_outputs.logits[:, prompt_ids.shape[-1]:]
            
        if logits.shape[-1] != teacher_logits.shape[-1]:
            teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
        
        completion_ids = sequences[:, prompt_ids.shape[-1]:]
        kl = compute_rkl(logits, teacher_logits, completion_ids, padding_id=self.processing_class.pad_token_id, reduction="mean")
        
        loss = kl.mean()
        
        return (loss, logits) if return_outputs else loss

## 2. 加载微调后的 Teacher 模型

In [ ]:
max_seq_length = 2048
dtype = None 
load_in_4bit = True
teacher, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/datasets/ningjingke/qwen-teacher-finetune/qwen_teacher_finetune",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# FastLanguageModel.for_inference(teacher)

## 3. 加载 Student 模型

In [ ]:
# 学生模型
student, _ = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-0.6B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

student = FastLanguageModel.get_peft_model(
    student,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none", # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False, # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)


student.print_trainable_parameters()

# 4. 数据加载

In [ ]:
import pandas as pd
import datasets
from torch.utils.data import Dataset
from transformers import DefaultDataCollator
from sklearn.model_selection import train_test_split

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction: 
You are an expert of film comment analysis. Analyze the given text from an online review and determine the sentiment polarity. Respond with 'positive' or 'negative'.   

### Question: 
{} 

### Response:
"""

class OPIMDBDataset(Dataset):
    def __init__(self, texts, tokenizer, max_prompt_length=512):
        super().__init__()
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_prompt_length = max_prompt_length
        self.padding_id = tokenizer.pad_token_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        prompt = alpaca_prompt.format(text)
        
        prompt_input_ids = self.tokenizer.encode(prompt)
        attention_mask = [1] * len(prompt_input_ids)
        
        # 截断或左填充到固定长度
        if len(prompt_input_ids) > self.max_prompt_length:
            prompt_input_ids = prompt_input_ids[-self.max_prompt_length:]
            attention_mask = attention_mask[-self.max_prompt_length:]
        else:
            pad_len = self.max_prompt_length - len(prompt_input_ids)
            prompt_input_ids = [self.padding_id] * pad_len + prompt_input_ids
            attention_mask = [0] * pad_len + attention_mask
            
        return {
            'input_ids': torch.tensor(prompt_input_ids),
            'attention_mask': torch.tensor(attention_mask)
        }


# 加载数据（需要 text 和 label）
train_full = pd.read_csv("/kaggle/input/datasets/ningjingke/corpus-imdb/labeledTrainData.tsv", header=0, delimiter="\t", quoting=3)

# 检查原始数据的类别分布
print("Original data sentiment distribution:")
print(train_full['sentiment'].value_counts())

# 按 6:2:2 划分，保留标签（使用 reset_index 确保索引连续）
# 第一次划分：80% (train+val) 和 20% (test)
train_val_df, test_df = train_test_split(train_full, test_size=0.2, random_state=42)
# 第二次划分：从 80% 中划分出 75% 的 train 和 25% 的 val，最终比例为 6:2:2
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

# 重置索引
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# 提取文本列表
train_texts = train_df['review'].tolist()
val_texts = val_df['review'].tolist()
test_texts = test_df['review'].tolist()

# 提取标签（数据中 sentiment 列已经是 1=positive, 0=negative，直接使用）
train_labels = train_df['sentiment'].tolist()
val_labels = val_df['sentiment'].tolist()
test_labels = test_df['sentiment'].tolist()

max_prompt_length = 1024
# 构建 on-policy 数据集
train_dataset = OPIMDBDataset(train_texts, tokenizer, max_prompt_length)
val_dataset = OPIMDBDataset(val_texts, tokenizer, max_prompt_length)
test_dataset_raw = OPIMDBDataset(test_texts, tokenizer, max_prompt_length)

print(f"\nTrain size: {len(train_texts)}")
print(f"Val size: {len(val_texts)}")
print(f"Test size: {len(test_texts)}")
print(f"Test labels: {len(test_labels)}")

# 检查各数据集的类别分布
print("\nTrain sentiment distribution:")
print(f"  Negative (0): {train_labels.count(0)}")
print(f"  Positive (1): {train_labels.count(1)}")

print("\nVal sentiment distribution:")
print(f"  Negative (0): {val_labels.count(0)}")
print(f"  Positive (1): {val_labels.count(1)}")

print("\nTest sentiment distribution:")
print(f"  Negative (0): {test_labels.count(0)}")
print(f"  Positive (1): {test_labels.count(1)}")

## 5. 设置训练参数

In [ ]:
# 训练参数
args = TrainingArguments(
    output_dir='results',
    num_train_epochs=0.1,
    do_train=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    logging_steps=100,
    save_strategy='epoch',
    save_total_limit=10,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    learning_rate=0.001,
    lr_scheduler_type='constant',
    optim="adamw_torch_fused",
    report_to = "none", 
)

trainer = OPDTrainer(
    model=student,
    teacher_model=teacher,
    tokenizer=tokenizer,
    train_dataset=train_dataset, 
    eval_dataset=val_dataset,
    data_collator=DefaultDataCollator(),
    args=args,
)

# 如果是初次训练resume_from_checkpoint为false，接着checkpoint继续训练，为True
trainer.train(resume_from_checkpoint=False)

# 三、模型预测

In [ ]:
from tqdm import tqdm
import numpy as np

# 提示模板 进行微调需要补充全部的推理指令
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction: 
You are an expert of film comment analysis. Analyze the given text from an online review and determine the sentiment polarity. Respond with 'positive' or 'negative'.   

### Question: 
{} 

### Response: 
"""


def format_test_prompt(examples):
    inputs = examples["review"]
    outputs = []
    for inp in inputs:
        text = prompt_style.format(inp)
        outputs.append(text)
    return {"text": outputs}

# 创建测试 Dataset
test_dict = {"review": test_texts, "id": list(range(len(test_texts)))}
test_dataset = datasets.Dataset.from_dict(test_dict)

formatted_test = test_dataset.map(format_test_prompt, batched=True)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=max_seq_length,
        add_special_tokens=True,
    )

tokenized_test = formatted_test.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)
tokenizer.padding_side = "left"

In [ ]:
student.eval()
student.to("cuda")

generation_config = {
    "max_new_tokens": 10,
    "do_sample": False,
    "temperature": 0.0,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

predictions = []
batch_size = 4
total_batches = (len(tokenized_test) + batch_size - 1) // batch_size

for i in tqdm(range(0, len(tokenized_test), batch_size), total=total_batches, desc="Inference"):
    batch = tokenized_test[i:i+batch_size]
    input_ids = [item for item in batch["input_ids"]]
    attention_mask = [item for item in batch["attention_mask"]]

    from transformers import DataCollatorForSeq2Seq
    collator = DataCollatorForSeq2Seq(tokenizer, padding=True, return_tensors="pt")
    batch_collated = collator([{"input_ids": ids, "attention_mask": mask} 
                               for ids, mask in zip(input_ids, attention_mask)])
    
    input_ids = batch_collated["input_ids"].to("cuda")
    attention_mask = batch_collated["attention_mask"].to("cuda")

    with torch.no_grad():
        outputs = student.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **generation_config
        )

    for j in range(len(outputs)):
        prompt_len = input_ids.shape[1]
        generated = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True).strip().lower()
        if "positive" in generated:
            pred = 1
        elif "negative" in generated:
            pred = 0
        else:
            words = generated.split()
            if words and "pos" in words[0]:
                pred = 1
            else:
                pred = 0
        predictions.append(pred)

test_pred = np.array(predictions)

# 计算准确率
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(test_labels, test_pred)
print(f"\n{'='*50}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*50}")

# 打印预测统计
unique_preds, pred_counts = np.unique(test_pred, return_counts=True)
print(f"\nPrediction distribution:")
for cls, count in zip(unique_preds, pred_counts):
    print(f"  Class {cls}: {count} samples ({count/len(test_pred)*100:.2f}%)")

# 打印真实标签统计
unique_true, true_counts = np.unique(test_labels, return_counts=True)
print(f"\nTrue label distribution:")
for cls, count in zip(unique_true, true_counts):
    print(f"  Class {cls}: {count} samples ({count/len(test_labels)*100:.2f}%)")

# 打印详细分类报告（如果预测有多个类别）
if len(unique_preds) > 1:
    print("\nClassification Report:")
    print(classification_report(test_labels, test_pred, target_names=['negative', 'positive']))
else:
    print("\nWarning: Model predicted only one class!")
    print("Classification report cannot be generated with single class predictions.")

# 打印混淆矩阵（处理单类别情况）
print("\nConfusion Matrix:")
cm = confusion_matrix(test_labels, test_pred, labels=[0, 1])  # 明确指定标签
print(cm)

# 安全地打印混淆矩阵各部分
if cm.shape == (2, 2):
    print(f"\nTrue Negatives: {cm[0,0]}")
    print(f"False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}")
    print(f"True Positives: {cm[1,1]}")
else:
    print(f"\nNote: Cannot print detailed confusion matrix due to single class")
    print(f"Matrix shape: {cm.shape}")

# 保存结果（修复 quoting 错误）
result_output = pd.DataFrame(data={"id": list(range(len(test_texts))), "review": test_texts, "true_label": test_labels, "pred_label": test_pred})
result_output.to_csv("/kaggle/working/decoder_distillation_with_accuracy.csv", index=False)
print(f"\nResults saved to: /kaggle/working/decoder_distillation_with_accuracy.csv")

# 显示一些生成的文本样本以便调试
print(f"\n{'='*50}")
print("Sample generated outputs (first 5):")
print(f"{'='*50}")
for i in range(min(5, len(test_texts))):
    print(f"\nSample {i}:")
    print(f"True label: {test_labels[i]}")
    print(f"Predicted: {test_pred[i]}")
    # 重新生成以查看实际输出
    sample_input = tokenizer(test_texts[i][:500], truncation=True, return_tensors="pt").to("cuda")
    sample_output = student.generate(**sample_input, max_new_tokens=20, do_sample=False)
    generated_text = tokenizer.decode(sample_output[0][sample_input['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"Generated: {generated_text}")